# 03 — Separación autor–afiliación

Esta libreta trabaja sobre `Canonico_Union_Trabajo.csv` y **no modifica el archivo original**.

El objetivo es separar cada autor con la afiliación que le corresponde.

Los dos archivos generados conservan exactamente las **16 columnas del modelo canónico**.

En `autor_afiliacion_separado.csv`:

- cada fila representa un autor individual de una publicación;
- `Autor_norm` contiene únicamente ese autor;
- `Afiliacion1` contiene la afiliación o afiliaciones asociadas a ese autor;
- `Afiliacion2` queda vacío porque ya no se conserva la estructura auxiliar específica de cada fuente;
- las otras columnas bibliográficas se conservan exactamente desde la fila original.

Si una fila original contiene al menos un autor cuya afiliación no puede resolverse con seguridad, toda la fila se aparta en `articulos_revision_manual.csv`, también con las 16 columnas originales.

In [1]:
import os
import pandas as pd
import re
import html
import unicodedata
import hashlib
from collections import defaultdict, Counter

# Rutas simples desde la carpeta notebooks
archivo = "../02_modelo_canonico/03_union/Canonico_Union_Trabajo.csv"
carpeta_salida = "../04_Limpieza/00_Separacion_Autor_Afiliacion"

salida = f"{carpeta_salida}/autor_afiliacion_separado.csv"
revision = f"{carpeta_salida}/articulos_revision_manual.csv"

os.makedirs(carpeta_salida, exist_ok=True)

def sha256(nombre_archivo):
    h = hashlib.sha256()
    with open(nombre_archivo, "rb") as f:
        for bloque in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloque)
    return h.hexdigest()

hash_antes = sha256(archivo)

df = pd.read_csv(
    archivo,
    dtype=str,
    keep_default_na=False,
    encoding="utf-8-sig"
)

columnas_esperadas = [
    "Base_origen",
    "Fuente_origen",
    "indice",
    "Titulo",
    "Año",
    "Autor_norm",
    "Afiliacion1",
    "Afiliacion2",
    "ISBN",
    "ISSN",
    "Doi",
    "URL",
    "Area",
    "SubArea",
    "Keywords",
    "Abstract"
]

if list(df.columns) != columnas_esperadas:
    raise ValueError("Las columnas no coinciden con las 16 columnas esperadas.")

if len(df) != 2153:
    raise ValueError(f"Número inesperado de filas: {len(df)}")

print("Filas originales:", len(df))
print("Columnas:", len(df.columns))

Filas originales: 2153
Columnas: 16


In [2]:
def decodificar_html(texto, max_iter=10):
    texto = "" if texto is None else str(texto)

    for _ in range(max_iter):
        nuevo = html.unescape(texto)
        if nuevo == texto:
            break
        texto = nuevo

    return texto


def separar(texto):
    texto = decodificar_html(texto)
    return [x.strip() for x in texto.split(";") if x.strip()]


SCOPUS_ID_RE = re.compile(r"\s*\((\d{7,12})\)\s*$")
EV_REF_RE = re.compile(r"\s*\(([\d,\s]+)\)\s*$")


def referencias_ev(autor):
    m = EV_REF_RE.search(decodificar_html(autor).strip())

    if not m:
        return []

    return [x.strip() for x in m.group(1).split(",") if x.strip()]


def mapa_afiliaciones_ev(texto):
    texto = decodificar_html(texto)
    patron = re.compile(r"(?:(?<=^)|(?<=\s))\((\d+)\)\s*")
    encontrados = list(patron.finditer(texto))

    resultado = {}

    for i, m in enumerate(encontrados):
        fin = encontrados[i + 1].start() if i + 1 < len(encontrados) else len(texto)
        resultado[m.group(1)] = texto[m.end():fin].strip(" ;")

    return resultado


def normalizar_doi(doi):
    doi = (doi or "").strip().lower()
    doi = re.sub(r"^https?://(?:dx\.)?doi\.org/", "", doi)
    doi = re.sub(r"^doi:\s*", "", doi)
    return re.sub(r"\s+", "", doi)


def normalizar_titulo(titulo):
    titulo = decodificar_html(titulo or "")
    titulo = "".join(
        c for c in unicodedata.normalize("NFKD", titulo)
        if not unicodedata.combining(c)
    )
    titulo = titulo.lower()
    titulo = re.sub(r"[^a-z0-9]+", " ", titulo)
    return " ".join(titulo.split())


def quitar_acentos(texto):
    return "".join(
        c for c in unicodedata.normalize("NFKD", texto)
        if not unicodedata.combining(c)
    )


def expandir_umlaut(texto):
    reemplazos = {
        "ä": "ae",
        "ö": "oe",
        "ü": "ue",
        "Ä": "Ae",
        "Ö": "Oe",
        "Ü": "Ue",
        "ß": "ss"
    }

    for a, b in reemplazos.items():
        texto = texto.replace(a, b)

    return quitar_acentos(texto)


def tokens_nombre(nombre, modo="acentos"):
    nombre = decodificar_html(nombre or "")
    nombre = SCOPUS_ID_RE.sub("", nombre)
    nombre = EV_REF_RE.sub("", nombre).strip()

    if "," in nombre:
        apellido, nombres = [x.strip() for x in nombre.split(",", 1)]
        partes = re.findall(r"[A-Za-zÀ-ÖØ-öø-ÿ]+", nombres)

        desarrolladas = []

        for parte in partes:
            if parte.isupper() and 2 <= len(parte) <= 4:
                desarrolladas.extend(list(parte))
            else:
                desarrolladas.append(parte)

        nombre = " ".join(desarrolladas + [apellido])

    if modo == "umlaut":
        nombre = expandir_umlaut(nombre)
    else:
        nombre = quitar_acentos(nombre)

    return [x.lower() for x in re.findall(r"[A-Za-z0-9]+", nombre) if x]


def token_compatible(a, b):
    return (
        a == b
        or (len(a) == 1 and b.startswith(a))
        or (len(b) == 1 and a.startswith(b))
    )


def comparar_tokens(a, b):
    if not a or not b:
        return False

    if a == b:
        return True

    if len(a) == len(b) and all(
        token_compatible(x, y) for x, y in zip(a, b)
    ):
        return True

    corto, largo = (a, b) if len(a) < len(b) else (b, a)

    if (
        len(corto) >= 2
        and token_compatible(corto[0], largo[0])
        and token_compatible(corto[-1], largo[-1])
    ):
        j = 0

        for token in corto:
            encontrado = False

            while j < len(largo):
                if token_compatible(token, largo[j]):
                    encontrado = True
                    j += 1
                    break

                j += 1

            if not encontrado:
                return False

        return True

    return False


def nombres_compatibles(a, b):
    for modo in ("acentos", "umlaut"):
        if comparar_tokens(tokens_nombre(a, modo), tokens_nombre(b, modo)):
            return True

    return False


def etiqueta_scopus_valida(autor, etiqueta):
    autor = SCOPUS_ID_RE.sub("", decodificar_html(autor)).strip()
    etiqueta = decodificar_html(etiqueta).strip()

    if "," not in autor:
        return nombres_compatibles(autor, etiqueta)

    apellido, nombres = [x.strip() for x in autor.split(",", 1)]

    def norm(texto):
        texto = quitar_acentos(texto).lower()
        texto = re.sub(r"[^a-z0-9]+", " ", texto)
        return " ".join(texto.split())

    apellido_n = norm(apellido)
    etiqueta_n = norm(etiqueta)

    if not (
        etiqueta_n == apellido_n
        or etiqueta_n.startswith(apellido_n + " ")
    ):
        return False

    resto = etiqueta_n[len(apellido_n):].strip().split()
    nombres_n = norm(nombres).split()

    iniciales = []

    for parte in resto:
        if len(parte) <= 4 and parte.isalpha():
            iniciales.extend(list(parte))
        else:
            iniciales.append(parte)

    return (
        len(iniciales) <= len(nombres_n)
        and all(n.startswith(i) for i, n in zip(iniciales, nombres_n))
    )


def corresponding_wos(texto):
    resultado = []

    for segmento in separar(texto):
        m = re.match(
            r"^(.*?)\s*\(corresponding author\)\s*,\s*(.*)$",
            segmento,
            re.I
        )

        if m:
            resultado.append((m.group(1).strip(), m.group(2).strip()))

    return resultado

In [3]:
# Primera pasada: separar autores y asociar afiliaciones según la fuente

registros = []

for _, fila in df.iterrows():
    fuente = fila["Fuente_origen"]
    autores = separar(fila["Autor_norm"])

    datos = {
        "Base_origen": fila["Base_origen"],
        "indice": fila["indice"],
        "Fuente_origen": fuente,
        "Titulo": fila["Titulo"],
        "Año": fila["Año"],
        "Doi": fila["Doi"]
    }

    # ACM / ProQuest / ScienceDirect
    if fuente in ("ACM", "ProQuest", "ScienceDirect"):
        if len(autores) != 1:
            raise ValueError(
                f"{fuente}: se esperaba un autor en "
                f"{fila['Base_origen']} + {fila['indice']}"
            )

        afiliaciones = [
            decodificar_html(x).strip()
            for x in (fila["Afiliacion1"], fila["Afiliacion2"])
            if x.strip()
        ]

        registros.append({
            **datos,
            "Autor_original": autores[0],
            "Afiliacion_asociada": " | ".join(dict.fromkeys(afiliaciones)),
            "Estado_separacion": "COMPLETO" if afiliaciones else "REVISAR",
            "Evidencia": f"Registro individual {fuente}",
            "Autor_id_fuente": "",
            "Referencias_faltantes": ""
        })

    # EV
    elif fuente == "EV":
        mapa = mapa_afiliaciones_ev(fila["Afiliacion1"])

        for autor in autores:
            refs = referencias_ev(autor)
            conocidas = [mapa[x] for x in refs if x in mapa]
            faltantes = [x for x in refs if x not in mapa]

            registros.append({
                **datos,
                "Autor_original": EV_REF_RE.sub("", autor).strip(),
                "Afiliacion_asociada": " | ".join(dict.fromkeys(conocidas)),
                "Estado_separacion": (
                    "COMPLETO" if refs and not faltantes else "INCOMPLETO"
                ),
                "Evidencia": "EV referencias numéricas de Afiliacion1",
                "Autor_id_fuente": "",
                "Referencias_faltantes": ",".join(faltantes)
            })

    # IEEE
    elif fuente == "IEEE":
        afiliaciones = separar(fila["Afiliacion1"])
        valido = len(autores) == len(afiliaciones)

        for posicion, autor in enumerate(autores):
            registros.append({
                **datos,
                "Autor_original": autor,
                "Afiliacion_asociada": afiliaciones[posicion] if valido else "",
                "Estado_separacion": "COMPLETO" if valido else "REVISAR",
                "Evidencia": (
                    "IEEE listas paralelas Autor_norm/Afiliacion1"
                    if valido
                    else "IEEE número de autores y afiliaciones distinto"
                ),
                "Autor_id_fuente": "",
                "Referencias_faltantes": ""
            })

    # Scopus
    elif fuente == "Scopus":
        entradas = separar(fila["Afiliacion2"])
        misma_longitud = len(autores) == len(entradas)

        for posicion, autor_original in enumerate(autores):
            m = SCOPUS_ID_RE.search(autor_original)
            scopus_id = m.group(1) if m else ""
            autor = SCOPUS_ID_RE.sub("", autor_original).strip()

            afiliacion = ""
            valido = False

            if misma_longitud:
                entrada = entradas[posicion]

                if "," in entrada:
                    etiqueta, afiliacion = entrada.split(",", 1)
                    valido = (
                        etiqueta_scopus_valida(autor_original, etiqueta)
                        and bool(afiliacion.strip())
                    )

            registros.append({
                **datos,
                "Autor_original": autor,
                "Afiliacion_asociada": afiliacion.strip() if valido else "",
                "Estado_separacion": "COMPLETO" if valido else "REVISAR",
                "Evidencia": (
                    "Scopus Afiliacion2 (etiqueta validada)"
                    if valido
                    else "Scopus entrada sin afiliación asociada"
                ),
                "Autor_id_fuente": scopus_id,
                "Referencias_faltantes": ""
            })

    # WoS
    elif fuente == "WoS":
        correspondientes = corresponding_wos(fila["Afiliacion2"])

        for autor in autores:
            afiliaciones = [
                afiliacion
                for etiqueta, afiliacion in correspondientes
                if nombres_compatibles(autor, etiqueta)
            ]

            registros.append({
                **datos,
                "Autor_original": autor,
                "Afiliacion_asociada": " | ".join(dict.fromkeys(afiliaciones)),
                "Estado_separacion": (
                    "PARCIAL_WOS" if afiliaciones else "PENDIENTE_WOS"
                ),
                "Evidencia": "WoS corresponding author" if afiliaciones else "",
                "Autor_id_fuente": "",
                "Referencias_faltantes": ""
            })

    else:
        raise ValueError(f"Fuente no contemplada: {fuente}")

print("Apariciones de autor extraídas:", len(registros))

Apariciones de autor extraídas: 12999


In [4]:
# Segunda pasada: recuperar afiliaciones desde otra representación
# de la misma publicación cuando la primera fuente no fue suficiente

prioridad = {
    "Scopus": 1,
    "EV": 2,
    "IEEE": 3,
    "ACM": 4,
    "ProQuest": 4,
    "ScienceDirect": 4
}

por_doi = defaultdict(list)
por_titulo_anio = defaultdict(list)

for r in registros:
    if (
        r["Fuente_origen"] != "WoS"
        and r["Estado_separacion"] == "COMPLETO"
        and r["Afiliacion_asociada"]
    ):
        doi = normalizar_doi(r["Doi"])

        if doi:
            por_doi[doi].append(r)

        clave = (
            normalizar_titulo(r["Titulo"]),
            str(r["Año"]).strip()
        )

        por_titulo_anio[clave].append(r)


for r in registros:
    if r["Estado_separacion"] == "COMPLETO":
        continue

    doi = normalizar_doi(r["Doi"])

    if doi:
        candidatos_base = por_doi.get(doi, [])
    else:
        clave = (
            normalizar_titulo(r["Titulo"]),
            str(r["Año"]).strip()
        )

        candidatos_base = por_titulo_anio.get(clave, [])

    candidatos = [
        x
        for x in candidatos_base
        if nombres_compatibles(r["Autor_original"], x["Autor_original"])
    ]

    if candidatos:
        mejor_prioridad = min(
            prioridad.get(x["Fuente_origen"], 99)
            for x in candidatos
        )

        mejores = [
            x
            for x in candidatos
            if prioridad.get(x["Fuente_origen"], 99) == mejor_prioridad
        ]

        identidad_unica = all(
            nombres_compatibles(
                mejores[0]["Autor_original"],
                x["Autor_original"]
            )
            for x in mejores[1:]
        )

        if identidad_unica:
            afiliaciones = list(dict.fromkeys(
                x["Afiliacion_asociada"]
                for x in mejores
                if x["Afiliacion_asociada"]
            ))

            if afiliaciones:
                r["Afiliacion_asociada"] = " | ".join(afiliaciones)
                r["Estado_separacion"] = "COMPLETO"
                r["Referencias_faltantes"] = ""

                nueva_evidencia = (
                    f"Misma publicación en {mejores[0]['Fuente_origen']}"
                )

                r["Evidencia"] = "; ".join(
                    x
                    for x in (r["Evidencia"], nueva_evidencia)
                    if x
                )

                continue

    if (
        r["Fuente_origen"] == "WoS"
        and r["Afiliacion_asociada"]
    ):
        r["Estado_separacion"] = "COMPLETO"


conteo = Counter(r["Estado_separacion"] for r in registros)
print(conteo)

Counter({'COMPLETO': 12158, 'INCOMPLETO': 837, 'REVISAR': 4})


In [5]:
# Separar las filas completamente resolubles de las que requieren revisión

resultado_total = pd.DataFrame(registros)

claves_problematicas = (
    resultado_total.loc[
        resultado_total["Estado_separacion"] != "COMPLETO",
        ["Base_origen", "indice"]
    ]
    .drop_duplicates()
)

claves_problematicas_set = set(
    map(
        tuple,
        claves_problematicas[["Base_origen", "indice"]].to_numpy()
    )
)

mascara_problema = resultado_total.apply(
    lambda r: (r["Base_origen"], r["indice"]) in claves_problematicas_set,
    axis=1
)

relaciones_resueltas = resultado_total.loc[~mascara_problema].copy()

if not (relaciones_resueltas["Estado_separacion"] == "COMPLETO").all():
    raise RuntimeError("La salida automática contiene relaciones no resueltas.")


# ---------------------------------------------------------
# CONSTRUIR LA SALIDA CON LAS 16 COLUMNAS CANÓNICAS
# ---------------------------------------------------------

# Recuperar todas las columnas originales de cada publicación.
resultado = relaciones_resueltas[
    ["Base_origen", "indice", "Autor_original", "Afiliacion_asociada"]
].merge(
    df,
    on=["Base_origen", "indice"],
    how="left"
)

# Una fila por autor.
resultado["Autor_norm"] = resultado["Autor_original"]

# Afiliacion1 pasa a contener exclusivamente la afiliación asociada
# al autor de esa fila.
resultado["Afiliacion1"] = resultado["Afiliacion_asociada"]

# Afiliacion2 era un campo auxiliar dependiente de la fuente.
# Después de separar autor-afiliación ya no debe conservar listas
# correspondientes a otros autores.
resultado["Afiliacion2"] = ""

# Conservar solamente las 16 columnas del modelo canónico.
resultado = resultado[columnas_esperadas].copy()


# ---------------------------------------------------------
# ARCHIVO PARA REVISIÓN MANUAL
# ---------------------------------------------------------

problemas = df.merge(
    claves_problematicas,
    on=["Base_origen", "indice"],
    how="inner"
)

# Las filas de revisión se conservan exactamente como estaban
# en Canonico_Union_Trabajo.
problemas = problemas[columnas_esperadas].copy()


# ---------------------------------------------------------
# VALIDACIONES
# ---------------------------------------------------------

if list(resultado.columns) != columnas_esperadas:
    raise RuntimeError(
        "autor_afiliacion_separado.csv no tiene exactamente las 16 columnas canónicas."
    )

if list(problemas.columns) != columnas_esperadas:
    raise RuntimeError(
        "articulos_revision_manual.csv no tiene exactamente las 16 columnas canónicas."
    )

if resultado["Autor_norm"].str.strip().eq("").any():
    raise RuntimeError("La salida automática contiene autores vacíos.")

if resultado["Afiliacion1"].str.strip().eq("").any():
    raise RuntimeError("La salida automática contiene afiliaciones vacías.")

if resultado[["Base_origen", "indice", "Autor_norm"]].duplicated().any():
    raise RuntimeError(
        "La salida automática contiene un autor duplicado dentro de la misma fila original."
    )


# Guardar resultados
resultado.to_csv(
    salida,
    index=False,
    encoding="utf-8-sig"
)

problemas.to_csv(
    revision,
    index=False,
    encoding="utf-8-sig"
)


# Comprobar que el archivo original no cambió
hash_despues = sha256(archivo)

if hash_antes != hash_despues:
    raise RuntimeError("Canonico_Union_Trabajo.csv fue modificado.")


# Validar que las 2153 filas originales quedaron clasificadas
# como resolubles o de revisión.
filas_resueltas = (
    df[["Base_origen", "indice"]]
    .merge(
        claves_problematicas,
        on=["Base_origen", "indice"],
        how="left",
        indicator=True
    )
    .query('_merge == "left_only"')
    [["Base_origen", "indice"]]
)

if len(filas_resueltas) + len(problemas) != len(df):
    raise RuntimeError(
        "Las filas resueltas + las filas de revisión no reconstruyen el total original."
    )


print()
print("RESUMEN FINAL")
print("-------------")
print("Filas originales:", len(df))
print("Filas originales completamente resolubles:", len(filas_resueltas))
print("Filas originales para revisión manual:", len(problemas))
print("Apariciones de autor totales:", len(resultado_total))
print("Filas generadas en autor_afiliacion_separado.csv:", len(resultado))
print("Columnas de autor_afiliacion_separado.csv:", len(resultado.columns))
print("Columnas de articulos_revision_manual.csv:", len(problemas.columns))
print("Archivo original intacto:", hash_antes == hash_despues)
print()
print("Salida automática:", salida)
print("Revisión manual:", revision)


RESUMEN FINAL
-------------
Filas originales: 2153
Filas originales completamente resolubles: 2127
Filas originales para revisión manual: 26
Apariciones de autor totales: 12999
Filas generadas en autor_afiliacion_separado.csv: 9515
Columnas de autor_afiliacion_separado.csv: 16
Columnas de articulos_revision_manual.csv: 16
Archivo original intacto: True

Salida automática: ../04_Limpieza/00_Separacion_Autor_Afiliacion/autor_afiliacion_separado.csv
Revisión manual: ../04_Limpieza/00_Separacion_Autor_Afiliacion/articulos_revision_manual.csv
